# Part 8 · Notebook 11 — Combining strategies and mean–variance optimization

**Sessions:** S21 (Combining strategies) · S22 (Mean–variance & covariance estimation) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Weight strategies by inverse volatility, and measure diversification.
2. Write the minimum-variance portfolio.
3. Shrink the covariance matrix, and see why sample-based mean–variance falls apart out of sample.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. Five strategies

The five synthetic strategies from notebook 03 (true Sharpes 0.8 to 0.3, vols 8% to 20%, pairwise correlation 0.2).

In [ ]:
S = p.strategy_returns()
display(pd.DataFrame({"vol": S.std() * np.sqrt(252), "Sharpe": S.apply(p.sharpe)}).round(3).T)
S.corr().round(2)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

**Inverse volatility:** `w_i ∝ 1/σ_i` (sample std, ddof=1), normalized to sum to 1. The simplest way to stop the most volatile strategy from dominating the book.

In [ ]:
def inverse_vol_weights(returns):
    iv = ...                                      # ✍️ 1 / std of each column
    return iv / iv.sum()

mine = p.attempt(inverse_vol_weights, S)
mine = p.check("inverse_vol_weights", mine, p.inverse_vol_weights(S))
cov = S.cov()
print(f"diversification ratio: 1/N {p.diversification_ratio(np.full(5, 0.2), cov):.2f}, inverse vol {p.diversification_ratio(mine, cov):.2f}")
mine.round(3)

The diversification ratio `Σ w_i σ_i / σ_p` is 1 for a single strategy; above 1.6 here, because correlations are low.

## 2. Minimum variance

The global minimum-variance portfolio (shorts allowed) is `Σ⁻¹1 / (1'Σ⁻¹1)`: solve `Σx = 1` with `np.linalg.solve`, then normalize. It needs only the covariance, no expected returns.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def min_variance(cov):
    cov = np.asarray(cov, dtype=float)
    x = ...                                       # ✍️ solve Σ x = 1
    return x / x.sum()

mine = p.attempt(min_variance, cov)
mine = p.check("min_variance", mine, p.min_variance(cov.to_numpy()))
pd.Series(mine, index=S.columns).round(3)

## 3. Why sample mean–variance fails

The maximum-Sharpe (tangency) portfolio `Σ⁻¹μ` needs **expected returns**, and sample means are extremely noisy (notebook 04). Fit it on each half of the history:

In [ ]:
h = len(S) // 2
pd.DataFrame({"first half": p.max_sharpe(S.iloc[:h].mean(), S.iloc[:h].cov()), "second half": p.max_sharpe(S.iloc[h:].mean(), S.iloc[h:].cov()),
              "true Sharpes": [0.8, 0.6, 0.5, 0.4, 0.3]}, index=S.columns).round(2)

Weights flip sign between halves: the optimizer is maximizing estimation error. The covariance is better estimated than the means, but with many assets and short windows it too is noisy. **Ledoit–Wolf shrinkage** pulls the sample covariance `S` toward a scaled identity `μI` (`μ` = the average variance), with an intensity `δ` estimated from the data (`p.shrinkage_intensity`): `Σ_shrunk = δ·μ·I + (1 − δ)·S`, where `S` is the sample covariance with divisor `T` (demeaned `X'X / T`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def ledoit_wolf(returns):
    X = np.asarray(returns, dtype=float)
    X = X - X.mean(0)
    S_ = X.T @ X / X.shape[0]
    mu = np.trace(S_) / S_.shape[0]
    d = p.shrinkage_intensity(returns)
    return ...                                    # ✍️

few = S.iloc[:60]                                 # three months of data: the regime where shrinkage matters
mine = [p.attempt(ledoit_wolf, S), p.attempt(ledoit_wolf, few)]
mine = p.check("ledoit_wolf", mine, [p.ledoit_wolf(S), p.ledoit_wolf(few)])
print(f"shrinkage intensity: 10 years of data {p.shrinkage_intensity(S):.3f}; 60 days {p.shrinkage_intensity(few):.3f}")

With ten years the data speaks for itself (little shrinkage); with 60 days the estimator leans heavily on the prior. Now the honest comparison: re-fit every month on the previous year and trade the next month (**walk-forward allocation**).

In [ ]:
alloc = {"1/N": lambda X: np.full(X.shape[1], 1 / X.shape[1]),
         "inverse vol": lambda X: p.inverse_vol_weights(X).to_numpy(),
         "min variance (sample)": lambda X: p.min_variance(X.cov().to_numpy()),
         "min variance (shrunk)": lambda X: p.min_variance(p.ledoit_wolf(X)),
         "max Sharpe (sample)": lambda X: p.max_sharpe(X.mean().to_numpy(), X.cov().to_numpy())}
rows = {}
for name, f in alloc.items():
    res = p.walk_forward_allocation(S, f)
    r = res["returns"]
    rows[name] = {"OOS Sharpe": p.sharpe(r), "OOS vol": r.std() * np.sqrt(252), "turnover per rebalance": res["turnover"]}
pd.DataFrame(rows).T.round(3)

Sample max-Sharpe explodes (a vol in the thousands of percent and huge turnover: the weights swing wildly from month to month). Simple 1/N and inverse vol are the ones to beat, as DeMiguel, Garlappi & Uppal (2009) found on real data.

## Wrap-up

* Start with 1/N and inverse vol; anything fancier must beat them **out of sample**, after turnover.
* Never feed sample means to an optimizer; shrink covariances.
* Graded version: `labs/part08/week30_portfolio` (long-only minimum variance with cvxpy).